## CleanUp

## Session Lifecycle Best Practices

AgentCore Runtime sessions consume memory (measured in GBHours) while active. Billing continues until the session is explicitly stopped or the idle timeout expires. There are two levels of cleanup:

- **Stop individual sessions** (`stop_runtime_session`): In production, use the data plane API to stop specific user sessions while keeping the runtime alive for other users. This immediately terminates the session's microVM and releases resources.
- **Delete the entire runtime** (`delete_agent_runtime`): Use the control plane API to tear down the entire runtime deployment, including all active sessions, endpoints, and infrastructure. This is appropriate for tutorial cleanup or decommissioning.

Cost management tips:
- **Configure idle timeout**: Set an appropriate idle timeout during session creation to automatically stop inactive sessions. Use shorter timeouts for development/testing and longer ones for production workloads.
- **Always clean up**: Wrap cleanup code in try/except blocks to ensure cleanup runs even if errors occur.
- **Delete runtime before other resources**: When tearing down, delete the runtime before IAM roles or ECR repositories to maintain proper authorization during teardown.
- **Use individual error handling**: Wrap each cleanup step in its own try/except block so that failure of one step doesn't prevent the others from executing.

**A2A multi-agent note**: This cleanup notebook tears down all three agent runtimes (AWS Docs, AWS Blogs, and Orchestrator) plus supporting resources. The cleanup ordering is: agent runtimes first (to stop billing), then IAM roles, Cognito, SSM parameters, and finally local files.

In [ ]:
%store -r

In [ ]:
from helpers.utils import (
    delete_agentcore_runtime_execution_role,
    delete_ssm_parameter,
    cleanup_cognito_resources,
    get_cognito_secret,
    delete_cognito_secret,
    runtime_resource_cleanup,
    delete_observability_resources,
    local_file_cleanup,
    ecr_repo_cleanup,
    short_memory_cleanup,
    AWS_DOCS_ROLE_NAME,
    ORCHESTRATOR_ROLE_NAME,
    AWS_BLOG_ROLE_NAME,
    SSM_DOCS_AGENT_ARN,
    SSM_BLOGS_AGENT_ARN,
)

print("✅ Dependencies imported successfully")

### Step 1: Delete Agent Runtimes

Delete all three agent runtimes first to stop GBHours billing. Each deletion is wrapped in its own try/except so that failure of one doesn't prevent the others from executing.

In [ ]:
from pathlib import Path
from bedrock_agentcore_starter_toolkit.operations.runtime.destroy import destroy_bedrock_agentcore

# Delete AWS Docs agent runtime
try:
    print("🚀 Deleting AWS Docs agent runtime...")
    destroy_bedrock_agentcore(
         config_path=Path(".bedrock_agentcore.yaml"),
         agent_name=MCP_AGENT_NAME,
         delete_ecr_repo=True
    )
    print("✅ AWS Docs agent runtime deleted")
except Exception as e:
    print(f"⚠️ Failed to delete AWS Docs agent runtime: {e}")

# Delete AWS Blogs agent runtime
try:
    print("🚀 Deleting AWS Blogs agent runtime...")
    destroy_bedrock_agentcore(
         config_path=Path(".bedrock_agentcore.yaml"),
         agent_name=BLOG_AGENT_NAME,
         delete_ecr_repo=True
    )
    print("✅ AWS Blogs agent runtime deleted")
except Exception as e:
    print(f"⚠️ Failed to delete AWS Blogs agent runtime: {e}")

# Delete Orchestrator agent runtime
try:
    print("🚀 Deleting Orchestrator agent runtime...")
    destroy_bedrock_agentcore(
         config_path=Path(".bedrock_agentcore.yaml"),
         agent_name=ORCHESTRATION_NAME,
         delete_ecr_repo=True
    )
    print("✅ Orchestrator agent runtime deleted")
except Exception as e:
    print(f"⚠️ Failed to delete Orchestrator agent runtime: {e}")

### Step 2: Delete IAM Roles and Security Resources

Now that runtimes are deleted, clean up IAM roles, Cognito resources, and SSM parameters. Each step is individually wrapped to ensure partial failures don't block remaining cleanup.

In [ ]:
import json

print("🛡️  Starting Security cleanup...")

# Delete AWS Docs execution role
try:
    print("  🗑️  Deleting Agent 1 - AWS Docs execution role...")
    delete_agentcore_runtime_execution_role(AWS_DOCS_ROLE_NAME)
    print("  ✅ AWS Docs execution role deleted")
except Exception as e:
    print(f"  ⚠️ Failed to delete AWS Docs execution role: {e}")

# Delete AWS Blogs execution role
try:
    print("  🗑️  Deleting Agent 2 - AWS Blogs execution role...")
    delete_agentcore_runtime_execution_role(AWS_BLOG_ROLE_NAME)
    print("  ✅ AWS Blogs execution role deleted")
except Exception as e:
    print(f"  ⚠️ Failed to delete AWS Blogs execution role: {e}")

# Delete Orchestrator execution role
try:
    print("  🗑️  Deleting Orchestration execution role...")
    delete_agentcore_runtime_execution_role(ORCHESTRATOR_ROLE_NAME)
    print("  ✅ Orchestrator execution role deleted")
except Exception as e:
    print(f"  ⚠️ Failed to delete Orchestrator execution role: {e}")

# Clean up Cognito resources
try:
    print("  🗑️  Cleaning up Cognito resources...")
    cs = json.loads(get_cognito_secret())
    cleanup_cognito_resources(cs['pool_id'])
    print("  ✅ Cognito resources cleaned up")
except Exception as e:
    print(f"  ⚠️ Failed to clean up Cognito resources: {e}")

# Delete Cognito secret
try:
    print("  🗑️  Deleting Cognito secret...")
    delete_cognito_secret()
    print("  ✅ Cognito secret deleted")
except Exception as e:
    print(f"  ⚠️ Failed to delete Cognito secret: {e}")

# Delete SSM parameters
try:
    print("  🗑️  Deleting SSM Parameter (Docs Agent ARN)...")
    delete_ssm_parameter(SSM_DOCS_AGENT_ARN)
    print("  ✅ SSM parameter deleted")
except Exception as e:
    print(f"  ⚠️ Failed to delete SSM parameter (Docs): {e}")

try:
    print("  🗑️  Deleting SSM Parameter (Blogs Agent ARN)...")
    delete_ssm_parameter(SSM_BLOGS_AGENT_ARN)
    print("  ✅ SSM parameter deleted")
except Exception as e:
    print(f"  ⚠️ Failed to delete SSM parameter (Blogs): {e}")

### Step 3: Clean Up Local Files and Observability Resources

In [ ]:
try:
    print("📁 Starting Local Files cleanup...")
    local_file_cleanup()
    print("✅ Local files cleaned up")
except Exception as e:
    print(f"⚠️ Failed to clean up local files: {e}")

In [ ]:
try:
    print("📊 Starting Observability cleanup for AWS Docs agent...")
    delete_observability_resources(MCP_AGENT_ID)
    print("✅ Observability resources deleted")
except Exception as e:
    print(f"⚠️ Failed to delete observability resources (Docs): {e}")

try:
    print("📊 Starting Observability cleanup for AWS Blogs agent...")
    delete_observability_resources(BLOG_AGENT_ID)
    print("✅ Observability resources deleted")
except Exception as e:
    print(f"⚠️ Failed to delete observability resources (Blogs): {e}")

try:
    print("📊 Starting Observability cleanup for Orchestrator...")
    delete_observability_resources(ORCHESTRATION_ID)
    print("✅ Observability resources deleted")
except Exception as e:
    print(f"⚠️ Failed to delete observability resources (Orchestrator): {e}")